In [1]:
import os
from dotenv import load_dotenv



In [3]:
# 1. Initialize Environment
load_dotenv()

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain



In [5]:
# --- CONFIGURATION ---
FILE_PATH = "datascience_and_analytics.pdf"
DB_PATH = "./rag_db"

# --- STEP 1: LOAD & SPLIT ---
loader = PyPDFLoader(FILE_PATH)
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
final_chunks = text_splitter.split_documents(docs)



In [6]:
# --- STEP 2: SETUP VECTOR SEARCH (CHROMA) ---
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=final_chunks,
    embedding=embeddings,
    persist_directory=DB_PATH
)
#

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8278.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
 #Semantic retriever (Dense)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# --- STEP 3: SETUP KEYWORD SEARCH (BM25) ---
# Keyword retriever (Sparse)
bm25_retriever = BM25Retriever.from_documents(final_chunks)
bm25_retriever.k = 3



In [13]:
# --- STEP 4: HYBRID ENSEMBLE ---
# We combine both. 'weights' controls the balance (0.5/0.5 is equal)
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever], 
    weights=[0.2, 0.8]
)



In [14]:
# --- STEP 5: BRAIN & PROMPT ---
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.3)

system_prompt = (
    "You are a professional assistant. Answer based ONLY on the provided context. "
    "The context is a mix of keyword matches and semantic matches.\n\n"
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])


In [15]:

# --- STEP 6: CHAIN & QUERY ---
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(hybrid_retriever, combine_docs_chain)


In [12]:

if __name__ == "__main__":
    query = "kdd"
    
    print(f"Running Hybrid Search for: {query}")
    result = rag_chain.invoke({"input": query})

    print("\n" + "="*50)
    print(f"HYBRID ANSWER:\n{result['answer']}")
    print("="*50)

Running Hybrid Search for: kdd

HYBRID ANSWER:
KDD is a process that involves several stages to extract meaningful insights from data. These stages include:

*   **Data Selection:** Gathering and filtering relevant data. For example, a fitness center might gather data from its membership system, focusing on the past six months of activity and filtering out inactive members.
*   **Data Cleaning and Preprocessing:** This is crucial in KDD to enhance data quality and improve data mining effectiveness. It involves correcting errors, handling missing values (e.g., filling gaps with the mean or most probable value), removing duplicates, and addressing noisy or outlier data (e.g., using techniques like binning, regression, or clustering).
*   **Data Transformation and Reduction:** This stage involves converting data into a format more suitable for analysis.
*   **Interpretation of Results:** Presenting insights in a meaningful and actionable way so decision-makers can use them to drive inform